# 03 Baseline Models (All Datasets)

This notebook trains baseline regressors (Linear Regression, Random Forest, XGBoost) on **all three datasets** (Desharnais, COCOMO-81, China) and saves per-dataset and combined metrics for comparison against CNN+PSO.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

root_dir = Path.cwd()
if not (root_dir / "src").exists() and (root_dir.parent / "src").exists():
    root_dir = root_dir.parent

if not (root_dir / "src").exists():
    raise FileNotFoundError("Could not find project root containing 'src' directory")

sys.path.insert(0, str(root_dir))

from src.evaluate import evaluate_predictions, save_metrics

processed_dir = root_dir / "data" / "processed"
metrics_dir = root_dir / "results" / "metrics"

# Load all 3 datasets
dataset_files = {
    "desharnais": "desharnais_processed.csv",
    "cocomo81": "cocomo81_processed.csv",
    "china": "china_processed.csv",
}

datasets = {}
for name, fname in dataset_files.items():
    path = processed_dir / fname
    if path.exists():
        datasets[name] = pd.read_csv(path)
        print(f"Loaded {name}: {datasets[name].shape}")
    else:
        print(f"WARNING: {fname} not found, skipping")

print(f"\nLoaded {len(datasets)} datasets")

Loaded desharnais: (81, 13)
Loaded cocomo81: (63, 20)
Loaded china: (499, 19)

Loaded 3 datasets


In [2]:
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("xgboost not installed — skipping XGBoost")


def get_baseline_models():
    """Return fresh (untrained) baseline models."""
    models = {
        "linear_regression": LinearRegression(),
        "random_forest": RandomForestRegressor(
            n_estimators=300, random_state=42, n_jobs=-1
        ),
    }
    if HAS_XGBOOST:
        models["xgboost"] = XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
        )
    return models

In [3]:
# Train baselines on EACH dataset
all_results = []

for ds_name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name} ({len(df)} samples, {df.shape[1]-1} features)")
    print(f"{'='*60}")

    target_col = df.columns[-1]
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    models = get_baseline_models()

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        row = evaluate_predictions(model_name, y_test, preds)
        row["dataset"] = ds_name
        all_results.append(row)
        print(f"  {model_name:20s} MAE={row['mae'].values[0]:10.2f}  RMSE={row['rmse'].values[0]:10.2f}  R²={row['r2'].values[0]:.4f}")

all_baselines = pd.concat(all_results, ignore_index=True)
# Reorder columns
cols = ["dataset", "model", "mae", "rmse", "r2", "mape", "mdmre", "pred25"]
all_baselines = all_baselines[cols].sort_values(["dataset", "rmse"])
all_baselines


Dataset: desharnais (81 samples, 12 features)
  linear_regression    MAE=   1435.05  RMSE=   1943.91  R²=0.7038
  random_forest        MAE=   1755.94  RMSE=   2283.39  R²=0.5914
  xgboost              MAE=   1693.69  RMSE=   2245.19  R²=0.6049

Dataset: cocomo81 (63 samples, 19 features)
  linear_regression    MAE=   1490.51  RMSE=   1921.08  R²=-10.9426
  random_forest        MAE=    236.48  RMSE=    404.22  R²=0.4712
  xgboost              MAE=    278.14  RMSE=    522.99  R²=0.1149

Dataset: china (499 samples, 18 features)
  linear_regression    MAE=    446.37  RMSE=   1296.14  R²=0.9441
  random_forest        MAE=    385.27  RMSE=   1605.22  R²=0.9143
  xgboost              MAE=    388.87  RMSE=   1503.30  R²=0.9248


,dataset,model,mae,rmse,r2,mape,mdmre,pred25
6,china,linear_regression,446.369674,1296.143319,0.944108,36.618911,0.112511,75.000000
8,china,xgboost,388.871586,1503.300169,0.924814,11.363155,0.072925,91.000000
7,china,random_forest,385.270333,1605.223998,0.914273,10.520690,0.078400,98.000000
4,cocomo81,random_forest,236.484410,404.224316,0.471247,198.270999,0.968000,15.384615
5,cocomo81,xgboost,278.135591,522.988085,0.114901,169.994311,0.774751,15.384615
3,cocomo81,linear_regression,1490.513248,1921.078032,-10.942580,5166.806104,24.503235,7.692308
0,desharnais,linear_regression,1435.054687,1943.914123,0.703827,54.219786,0.296057,41.176471
2,desharnais,xgboost,1693.691995,2245.187760,0.604909,57.648496,0.404122,23.529412
1,desharnais,random_forest,1755.944118,2283.387200,0.591351,77.443114,0.343604,23.529412


In [4]:
# Save combined baseline metrics
metrics_path = save_metrics(
    all_baselines,
    file_name="baseline_metrics.csv",
    output_dir=metrics_dir,
)
print("Saved baseline metrics to:", metrics_path)

# Also save per-dataset files for easy lookup
for ds_name in datasets:
    ds_metrics = all_baselines[all_baselines["dataset"] == ds_name]
    save_metrics(ds_metrics, file_name=f"baseline_{ds_name}_metrics.csv", output_dir=metrics_dir)
    print(f"  Saved baseline_{ds_name}_metrics.csv")

Saved baseline metrics to: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\results\metrics\baseline_metrics.csv
  Saved baseline_desharnais_metrics.csv
  Saved baseline_cocomo81_metrics.csv
  Saved baseline_china_metrics.csv
